# O rețea neuronală completă

Construim un multilayer perceptron pentru o problemă neliniară și tratăm bucla de antrenare ca pe un experiment măsurabil: separarea datelor, mini-batch-uri, regularizare și moduri distincte de train/eval.

## 1. Date și separare

Generăm două clase concentrice. Separarea train/validation trebuie făcută înainte de orice transformare care învață statistici. Altfel, informația din validare se scurge în antrenare și estimarea generalizării devine optimistă.

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(21)
n = 1000
X = torch.randn(n, 2)
y = ((X[:,0]**2 + X[:,1]**2) > 1.4).long()
perm = torch.randperm(n)
train_idx, val_idx = perm[:800], perm[800:]
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
print('train:', X_train.shape, 'validation:', X_val.shape)

train: torch.Size([800, 2]) validation: torch.Size([200, 2])


## 2. Arhitectura și numărul de parametri

Modelul calculează $f(x)=W_2\operatorname{ReLU}(W_1x+b_1)+b_2$. Stratul ascuns proiectează planul într-un spațiu cu 32 de dimensiuni, unde o frontieră liniară poate aproxima cercul. Numărul parametrilor este $2\cdot32+32+32\cdot2+2=162$.

In [2]:
model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)
print('parametri:', sum(p.numel() for p in model.parameters()))

parametri: 162


## 3. Logits și cross-entropy

`CrossEntropyLoss` primește logits ne-normalizați și etichete întregi. Intern combină `log_softmax` cu negative log-likelihood într-o formă stabilă numeric:

$$L=-\frac1B\sum_i\log\frac{e^{z_{i,y_i}}}{\sum_k e^{z_{i,k}}}. $$

Nu adăugăm `Softmax` în model în timpul antrenării.

In [3]:
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)

for epoch in range(1, 21):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()*len(xb)
    if epoch == 1 or epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(X_val).argmax(1) == y_val).float().mean()
        print(f'epoca {epoch:02d} | loss {total_loss/len(X_train):.3f} | val acc {acc:.3f}')

epoca 01 | loss 0.641 | val acc 0.745
epoca 10 | loss 0.246 | val acc 0.945
epoca 20 | loss 0.126 | val acc 0.970


## 4. Diagnosticarea antrenării

Urmărim separat loss-ul de train și metricile de validare. Dacă train loss scade iar validation loss crește, modelul supraînvață. Dacă ambele rămân mari, verificăm rata de învățare, normalizarea intrărilor, capacitatea și corectitudinea etichetelor.

`model.train()` și `model.eval()` schimbă comportamentul modulelor precum Dropout și BatchNorm; `torch.no_grad()` controlează înregistrarea grafului, nu modul modulelor.

## Proiect

1. Adaugă curbele train/validation loss și identifică epoca optimă.
2. Compară SGD cu momentum și AdamW la același buget de pași.
3. Adaugă BatchNorm și explică diferența dintre statisticile de train și eval.
4. Salvează `state_dict`, reconstruiește modelul și verifică identitatea predicțiilor.
5. Extinde problema la trei inele și discută matricea de confuzie.